Implement Scaled Dot-Product Attention and a Mini Transformer Encoder 
You are required to implement the core components of the Transformer architecture 
using PyTorch. The assignment focuses on understanding how attention works inside a 
Transformer. 
Problem Statement 
Implement a simple Transformer Encoder from scratch and test it on a small text 
classification task. 

You should implement the following: 
1. Scaled Dot-Product Attention  
2. Multi-Head Self-Attention  
3. Positional Encoding  
4. Feed Forward Network  
5. Transformer Encoder Block  
6. A simple classifier using the Transformer Encoder  

Dataset 
Use any small dataset, such as:

• IMDB small subset  
• AG News subset  
• SMS Spam Detection dataset  
• A custom dataset with 2 classes, for example positive/negative sentences  

You may also create a small toy dataset manually.

<div align="center">
----------------------------------------------------------------------------------------------XXXXXXXXXXXXXXXXXXXXXX----------------------------------------------------------------------------------------------
</div>

In [19]:
# import kagglehub
# path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
# print("Path to dataset files:", path)

In [48]:
#import necessary libraries
import pandas as pd
from collections import Counter
import re

In [30]:
DATASET_PATH = r'D:\machine_learning\datasets\sms_spam_detection'
df = pd.read_csv(DATASET_PATH + '\spam.csv', encoding='latin-1') #utf-8 is failing for some reason. okay there is something wrong in row number 2, escape character probably
display(df)

df = df[['v1', 'v2']] 
df.columns = ['label', 'text']

# replacing labels with 1 and 0, with spam being 1 and ham being 0
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# final dataset for training/testing
display(df)

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Sumed\AppData\Local\Temp\ipykernel_10452\2225014558.py:2: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(DATASET_PATH + '\spam.csv', encoding='latin-1') #utf-8 is failing for some reason. okay there is something wrong in row number 2, escape character probably


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


TASK 1. Preprocess the text data:  
o Tokenize sentences  
o Convert words into integer IDs  
o Pad sequences to the same length

In [52]:
#part a - Tokenize Sentences

#tokenizing process
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text) #removing special characters and punctuation to simplify vocabulary
    return text.split()
df['tokens'] = df['text'].apply(tokenize) # apply for each row of the dataframe
display(df[['text', 'tokens']])

,text,tokens
0,"Go until jurong point, crazy.. Available only ...","[go, until, jurong, point, crazy, available, o..."
1,Ok lar... Joking wif u oni...,"[ok, lar, joking, wif, u, oni]"
2,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f..."
3,U dun say so early hor... U c already then say...,"[u, dun, say, so, early, hor, u, c, already, t..."
4,"Nah I don't think he goes to usf, he lives aro...","[nah, i, dont, think, he, goes, to, usf, he, l..."
...,...,...
5567,This is the 2nd time we have tried 2 contact u...,"[this, is, the, 2nd, time, we, have, tried, 2,..."
5568,Will Ì_ b going to esplanade fr home?,"[will, b, going, to, esplanade, fr, home]"
5569,"Pity, * was in mood for that. So...any other s...","[pity, was, in, mood, for, that, soany, other,..."
5570,The guy did some bitching but I acted like i'd...,"[the, guy, did, some, bitching, but, i, acted,..."


In [54]:
#part b - Convert words into integer IDs

word_counter = Counter()
for tokens in df['tokens']:
    word_counter.update(tokens)

# create vocabulary
vocab = {'<PAD>': 0
        ,'<UNK>': 1
        }

# assign integer IDs
for word in word_counter:
    vocab[word] = len(vocab)

print("Vocabulary Size:", len(vocab)) #13498

# we do this because transformers require equal sequence lengths
def tokens_to_ids(tokens, vocab):
    return [vocab.get(token, vocab['<UNK>']) for token in tokens]

df['input_ids'] = df['tokens'].apply(lambda x: tokens_to_ids(x, vocab))

display(df[['tokens', 'input_ids']])

Vocabulary Size: 9479


,tokens,input_ids
0,"[go, until, jurong, point, crazy, available, o...","[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1..."
1,"[ok, lar, joking, wif, u, oni]","[22, 23, 24, 25, 26, 27]"
2,"[free, entry, in, 2, a, wkly, comp, to, win, f...","[28, 29, 9, 30, 31, 32, 33, 34, 35, 36, 37, 38..."
3,"[u, dun, say, so, early, hor, u, c, already, t...","[26, 51, 52, 53, 54, 55, 26, 56, 57, 58, 52]"
4,"[nah, i, dont, think, he, goes, to, usf, he, l...","[59, 60, 61, 62, 63, 64, 34, 65, 63, 66, 67, 6..."
...,...,...
5567,"[this, is, the, 2nd, time, we, have, tried, 2,...","[171, 101, 155, 399, 477, 369, 132, 436, 30, 8..."
5568,"[will, b, going, to, esplanade, fr, home]","[222, 261, 278, 34, 1970, 1339, 166]"
5569,"[pity, was, in, mood, for, that, soany, other,...","[9475, 358, 9, 5535, 88, 276, 9476, 1391, 9477]"
5570,"[the, guy, did, some, bitching, but, i, acted,...","[155, 2922, 250, 84, 9478, 368, 60, 8571, 83, ..."


In [51]:
#part c - pad sequences to the same length

#decide max length
df['length'] = df['input_ids'].apply(len)
print(df['length'].describe())


#most messages are short in length, keeping max_len as 20, might change later
MAX_LEN = 20
def pad_sequence(sequence, max_len):
    if len(sequence) < max_len:
        sequence = sequence + [0] * (max_len - len(sequence))
    else:
        sequence = sequence[:max_len]
    return sequence

df['padded_input_ids'] = df['input_ids'].apply(lambda x: pad_sequence(x, MAX_LEN))

display(df[['input_ids', 'padded_input_ids']])

count    5572.000000
mean       15.238514
std        11.085235
min         0.000000
25%         7.000000
50%        12.000000
75%        22.000000
max       171.000000
Name: length, dtype: float64


,input_ids,padded_input_ids
0,"[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1...","[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1..."
1,"[22, 23, 24, 25, 26, 27]","[22, 23, 24, 25, 26, 27, 0, 0, 0, 0, 0, 0, 0, ..."
2,"[28, 29, 9, 30, 31, 32, 33, 34, 35, 36, 37, 38...","[28, 29, 9, 30, 31, 32, 33, 34, 35, 36, 37, 38..."
3,"[26, 51, 52, 53, 54, 55, 26, 56, 57, 58, 52]","[26, 51, 52, 53, 54, 55, 26, 56, 57, 58, 52, 0..."
4,"[59, 60, 61, 62, 63, 64, 34, 65, 63, 66, 67, 6...","[59, 60, 61, 62, 63, 64, 34, 65, 63, 66, 67, 6..."
...,...,...
5567,"[171, 101, 155, 399, 477, 369, 132, 436, 30, 8...","[171, 101, 155, 399, 477, 369, 132, 436, 30, 8..."
5568,"[222, 261, 278, 34, 1970, 1339, 166]","[222, 261, 278, 34, 1970, 1339, 166, 0, 0, 0, ..."
5569,"[9475, 358, 9, 5535, 88, 276, 9476, 1391, 9477]","[9475, 358, 9, 5535, 88, 276, 9476, 1391, 9477..."
5570,"[155, 2922, 250, 84, 9478, 368, 60, 8571, 83, ...","[155, 2922, 250, 84, 9478, 368, 60, 8571, 83, ..."


TASK 2. Implement Scaled Dot-Product Attention using the formula:  

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$